<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/ESAA_1113_%EA%B3%BC%EC%A0%9C2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 문자 단위 RNN(Char RNN): 실습 2

In [1]:
# 문자 단위 RNN(Char RNN)

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [2]:
# 훈련 데이터 전처리하기
input_str = 'apple'
label_str = 'pple!'
char_vocab = sorted(list(set(input_str+label_str)))
vocab_size = len(char_vocab)
print ('문자 집합의 크기 : {}'.format(vocab_size))

문자 집합의 크기 : 5


In [3]:
input_size = vocab_size # 입력의 크기는 문자 집합의 크기
hidden_size = 5
output_size = 5
learning_rate = 0.1

In [4]:
char_to_index = dict((c, i) for i, c in enumerate(char_vocab)) # 문자에 고유한 정수 인덱스 부여
print(char_to_index)

{'!': 0, 'a': 1, 'e': 2, 'l': 3, 'p': 4}


In [5]:
# 정수로부터 문자 얻기
index_to_char={}
for key, value in char_to_index.items():
    index_to_char[value] = key
print(index_to_char)

{0: '!', 1: 'a', 2: 'e', 3: 'l', 4: 'p'}


In [6]:
# 입력 데이터, 레이블 데이터의 각 문자들 정수로 맴핑
x_data = [char_to_index[c] for c in input_str]
y_data = [char_to_index[c] for c in label_str]
print(x_data)
print(y_data)

[1, 4, 4, 3, 2]
[4, 4, 3, 2, 0]


In [9]:
# 입력 시퀀스의 각 문자들을 원-핫 벡터로 변환
x_one_hot = [np.eye(vocab_size)[x] for x in x_data]
print(x_one_hot)

[array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.]])]


In [10]:
# 파이토치의 nn.RNN() - 입력: 3차원 텐서
# 배치 차원 추가
# 텐서 연산인 unsqueeze(0)를 통해 해결할 수도 있었음.
x_data = [x_data]
y_data = [y_data]
print(x_data)
print(y_data)

[[[1, 4, 4, 3, 2]]]
[[[4, 4, 3, 2, 0]]]


In [11]:
# 입력 데이터, 레이블 데이터를 텐서로 변환
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

/tmp/ipykernel_1125/533076003.py:2: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:252.)
  X = torch.FloatTensor(x_one_hot)


In [12]:
# 각 텐서의 크기 확인

print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))

훈련 데이터의 크기 : torch.Size([1, 5, 5])
레이블의 크기 : torch.Size([1, 1, 5])


# 2) 모델 구현하기

fc: 완전 연결층(fully-connected layer)

: 출력층으로 사용

In [13]:
class Net(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_size, hidden_size, batch_first=True) # RNN 셀 구현
        self.fc = torch.nn.Linear(hidden_size, output_size, bias=True) # 출력층 구현

    def forward(self, x): # 구현한 RNN 셀과 출력층을 연결
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x

In [14]:
# 클래스로 정의한 모델을 net에 저장
net = Net(input_size, hidden_size, output_size)

In [15]:
# 입력된 모델에 입력을 넣어서 출력의 크기 확인
outputs = net(X)
print(outputs.shape) # 3차원 텐서


torch.Size([1, 5, 5])


In [16]:
# niew를 이용하여 배치 차원, 시점 차원을 하나로 생성

print(outputs.view(-1, input_size).shape) # 2차원 텐서로 변환

torch.Size([5, 5])


In [17]:
# 레이블 데이터의 크기 다시
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([1, 1, 5])
torch.Size([5])


- 레이블 데이터의 크기는 (1,5). 정확도 측정 시, 이를 펼쳐서 계산. -> 크기 (5)

In [18]:
# 옵티마이저, 손실 함수 정의
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [19]:
# 총 100번의 에포크 학습
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X)
    loss = criterion(outputs.view(-1, input_size), Y.view(-1)) # view를 하는 이유는 Batch 차원 제거를 위해
    loss.backward() # 기울기 계산
    optimizer.step() # 아까 optimizer 선언 시 넣어둔 파라미터 업데이트

    # 아래 세 줄은 모델이 실제 어떻게 예측했는지를 확인하기 위한 코드.
    result = outputs.data.numpy().argmax(axis=2) # 최종 예측값인 각 time-step 별 5차원 벡터에 대해서 가장 높은 값의 인덱스를 선택
    result_str = ''.join([index_to_char[c] for c in np.squeeze(result)])
    print(i, "loss: ", loss.item(), "prediction: ", result, "true Y: ", y_data, "prediction str: ", result_str)

0 loss:  2.05084228515625 prediction:  [[1 1 1 1 1]] true Y:  [[[4, 4, 3, 2, 0]]] prediction str:  aaaaa
1 loss:  1.6599862575531006 prediction:  [[0 0 0 0 0]] true Y:  [[[4, 4, 3, 2, 0]]] prediction str:  !!!!!
2 loss:  1.4991910457611084 prediction:  [[0 0 0 0 0]] true Y:  [[[4, 4, 3, 2, 0]]] prediction str:  !!!!!
3 loss:  1.3741885423660278 prediction:  [[0 0 0 0 0]] true Y:  [[[4, 4, 3, 2, 0]]] prediction str:  !!!!!
4 loss:  1.175755262374878 prediction:  [[0 4 3 2 0]] true Y:  [[[4, 4, 3, 2, 0]]] prediction str:  !ple!
5 loss:  0.9583357572555542 prediction:  [[4 3 3 2 0]] true Y:  [[[4, 4, 3, 2, 0]]] prediction str:  plle!
6 loss:  0.7647858262062073 prediction:  [[4 3 3 2 0]] true Y:  [[[4, 4, 3, 2, 0]]] prediction str:  plle!
7 loss:  0.5935876965522766 prediction:  [[4 3 3 2 0]] true Y:  [[[4, 4, 3, 2, 0]]] prediction str:  plle!
8 loss:  0.4474160671234131 prediction:  [[4 4 3 2 0]] true Y:  [[[4, 4, 3, 2, 0]]] prediction str:  pple!
9 loss:  0.34208670258522034 prediction:

# 2. 더 많은 데이터로 학습하 문자 단위 RNN(Char RNN)


In [20]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1) 훈련 데이터 전처리하기

In [21]:
sentence = ("if you want to build a ship, don't drum up people together to "
            "collect wood and don't assign them tasks and work, but rather "
            "teach them to long for the endless immensity of the sea.")

In [22]:
# 문자 집합 생성
# 각 문자에 고유한 정수 부여
char_set = list(set(sentence)) # 중복을 제거한 문자 집합 생성
char_dic = {c: i for i, c in enumerate(char_set)} # 각 문자에 정수 인코딩

In [23]:
print(char_dic) # 공백도 여기서는 하나의 원소

{'e': 0, 'o': 1, '.': 2, 'i': 3, 'b': 4, 'u': 5, 'f': 6, ',': 7, 's': 8, 'l': 9, 'm': 10, 'h': 11, "'": 12, 'c': 13, 'w': 14, 't': 15, 'y': 16, 'g': 17, ' ': 18, 'p': 19, 'k': 20, 'a': 21, 'r': 22, 'n': 23, 'd': 24}


- 각 문서에 정수 부여

- 25개의 문자 존재

In [24]:
dic_size = len(char_dic)
print('문자 집합의 크기 : {}'.format(dic_size))

문자 집합의 크기 : 25


- 문자 집합의 크기: 25 = 매 시점마다 들어갈 입력의 크기

In [25]:
# 하이퍼파라미터 설정
hidden_size = dic_size
sequence_length = 10  # 임의 숫자 지정
learning_rate = 0.1

In [26]:
# 임의로 지정한 sequence_length 값(=10)의 단위로 샘플들을 잘라서 데이터 생성
# 데이터 구성
x_data = []
y_data = []

for i in range(0, len(sentence) - sequence_length):
    x_str = sentence[i:i + sequence_length]
    y_str = sentence[i + 1: i + sequence_length + 1]
    print(i, x_str, '->', y_str)

    x_data.append([char_dic[c] for c in x_str])  # x str to index
    y_data.append([char_dic[c] for c in y_str])  # y str to index

0 if you wan -> f you want
1 f you want ->  you want 
2  you want  -> you want t
3 you want t -> ou want to
4 ou want to -> u want to 
5 u want to  ->  want to b
6  want to b -> want to bu
7 want to bu -> ant to bui
8 ant to bui -> nt to buil
9 nt to buil -> t to build
10 t to build ->  to build 
11  to build  -> to build a
12 to build a -> o build a 
13 o build a  ->  build a s
14  build a s -> build a sh
15 build a sh -> uild a shi
16 uild a shi -> ild a ship
17 ild a ship -> ld a ship,
18 ld a ship, -> d a ship, 
19 d a ship,  ->  a ship, d
20  a ship, d -> a ship, do
21 a ship, do ->  ship, don
22  ship, don -> ship, don'
23 ship, don' -> hip, don't
24 hip, don't -> ip, don't 
25 ip, don't  -> p, don't d
26 p, don't d -> , don't dr
27 , don't dr ->  don't dru
28  don't dru -> don't drum
29 don't drum -> on't drum 
30 on't drum  -> n't drum u
31 n't drum u -> 't drum up
32 't drum up -> t drum up 
33 t drum up  ->  drum up p
34  drum up p -> drum up pe
35 drum up pe -> rum up peo
36

- 170개의 샘플 생성

- 각 샘플의 각 문자들은 고유한 정수로 인코딩이 된 상

In [27]:
# 첫 번째 샘플의 입력 데이터, 레이블 데이터 출력
print(x_data[0])
print(y_data[0])

[3, 6, 18, 16, 1, 5, 18, 14, 21, 23]
[6, 18, 16, 1, 5, 18, 14, 21, 23, 15]


In [28]:
# 입력 시퀀스에 대한 원-핫 인코딩 수행
# 입력 데이터, 레이블 데이터를 텐서로 변환

x_one_hot = [np.eye(dic_size)[x] for x in x_data] # x 데이터는 원-핫 인코딩
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)


In [29]:
# 훈련 데이터, 레이블 데이터의 크기 확인
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))

훈련 데이터의 크기 : torch.Size([170, 10, 25])
레이블의 크기 : torch.Size([170, 10])


In [30]:
# 원-핫 인코딩 된 결과를 보기 위해 첫번째 샘플만 출력
print(X[0])

tensor([[0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,

In [31]:
# 레이블 데이터의 첫번째 샘플 출력
print(Y[0])


tensor([ 6, 18, 16,  1,  5, 18, 14, 21, 23, 15])


- 위 레이블 시퀀스: f you want



# 2) 모델 구현하기

은닉층 두 개 쌓기

In [32]:
class Net(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, layers): # 현재 hidden_size는 dic_size와 같음.
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_dim, hidden_dim, num_layers=layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, hidden_dim, bias=True)

    def forward(self, x):
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x

In [33]:
net = Net(dic_size, hidden_size, 2) # 이번에는 층을 두 개 쌓습니다.

- nn.RNN() 안에 num_layers 인자 사용 -> 은닉층을 몇 개 쌓을 것인지 의미





In [34]:
# 비용 함수, 옵티마이저
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [35]:
# 모델에 입력을 넣어서, 출력 크기 확인
outputs = net(X)
print(outputs.shape) # 3차원 텐서

torch.Size([170, 10, 25])


- (170, 10, 25): 각각 배치 차원, 시점(timesteps), 출력의 크기

- 정확도 측정 시, 이를 모두 펼쳐서 계산

(view로 치 차원, 시점 차원 하나로 생성)

In [36]:
print(outputs.view(-1, dic_size).shape) # 2차원 텐서로 변환.

torch.Size([1700, 25])


In [37]:
# 레이블 데이터의 크기
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([170, 10])
torch.Size([1700])


In [38]:
# 옵티마이저, 손실 함수 정의
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X) # (170, 10, 25) 크기를 가진 텐서를 매 에포크마다 모델의 입력으로 사용
    loss = criterion(outputs.view(-1, dic_size), Y.view(-1))
    loss.backward()
    optimizer.step()

    # results의 텐서 크기는 (170, 10)
    results = outputs.argmax(dim=2)
    predict_str = ""
    for j, result in enumerate(results):
        if j == 0: # 처음에는 예측 결과를 전부 가져오지만
            predict_str += ''.join([char_set[t] for t in result])
        else: # 그 다음에는 마지막 글자만 반복 추가
            predict_str += char_set[result[-1]]

    print(predict_str)

appswawwwswwwwwwwwwwppwpwpwwpspawwwsaaapswwwpwwwpwwwaawapwwwwawswwswwwwwsawpaawswawwwwapppaawwpwpwwwappwswwpwswwwwwwwwwwwpwwwawwawwwpwpwwwwwaawppawwwpwsawwpwappwwpaapwwawawwwpwsaw
                                                                                                                                                                                   
                                                                                                                                                                                   
sf.aeg.aegegndgngngngngngngngngnhgghennlglllglhehlhdlhgdnegnlghnlgelnghndllgnlhndlgnndgnnlghndnhnglglngledlnddghndghlgdhgnlhnnngnghgnnggnahnggehnglnlntnnltnnggnlngdghtngggngdgeonl
naoaoeeeoeeeoeaeeeaeaoaeaoaeaoeeeeeeeoeeoeoeoeeeeooeeeeeeeaoeeeeeeeeoaoeeeeoeeeoeeeoeoeeoeeeoeeeeeaeeeaoaeeoeeeeooeeoeeeeeeeoeeeeoeeoeeoeeoeeeeoeeeoeoeoeheeeeaoaeaeeeaoeoaeaoaeeee
ttotooeooteeooeooeeeaotoettetoeetotoeoetoeoeeeeeetoeeeottetoteoetoeootootetoieeotteetoeettetoeeetoto

- 처음에는 이상한 예측

- 마지막 에포크에서 꽤 정확한 문자 생성